# Bases de datos · SQLite y SQL

SQL con el módulo estándar `sqlite3` sobre una base de datos de una tienda (`tienda.db`) que crea el propio notebook: clientes, productos, pedidos y líneas de pedido.

## 1. Crear el esquema

Crear las tablas `clientes`, `productos`, `pedidos` y `lineas` con claves primarias, claves foráneas y restricciones `CHECK` (precio positivo, stock no negativo, cantidad positiva). Activar las claves foráneas y comprobar que SQLite rechaza un pedido de un cliente que no existe.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

ruta = Path("tienda.db")
ruta.unlink(missing_ok=True)
con = sqlite3.connect(ruta)
con.execute("PRAGMA foreign_keys = ON")  # SQLite las ignora si no se activan en cada conexión
con.executescript('''
CREATE TABLE clientes (
    id     INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    ciudad TEXT NOT NULL,
    alta   TEXT NOT NULL
);
CREATE TABLE productos (
    id        INTEGER PRIMARY KEY,
    nombre    TEXT NOT NULL UNIQUE,
    categoria TEXT NOT NULL,
    precio    REAL NOT NULL CHECK (precio > 0),
    stock     INTEGER NOT NULL CHECK (stock >= 0)
);
CREATE TABLE pedidos (
    id         INTEGER PRIMARY KEY,
    cliente_id INTEGER NOT NULL REFERENCES clientes (id),
    fecha      TEXT NOT NULL
);
CREATE TABLE lineas (
    pedido_id   INTEGER NOT NULL REFERENCES pedidos (id) ON DELETE CASCADE,
    producto_id INTEGER NOT NULL REFERENCES productos (id),
    cantidad    INTEGER NOT NULL CHECK (cantidad > 0),
    precio      REAL NOT NULL,
    PRIMARY KEY (pedido_id, producto_id)
);
''')
print([fila[0] for fila in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")])
print(con.execute("PRAGMA foreign_key_list(lineas)").fetchall())

try:
    con.execute("INSERT INTO pedidos (cliente_id, fecha) VALUES (99, '2026-01-01')")
except sqlite3.IntegrityError as e:
    print("Rechazado:", e)

['clientes', 'productos', 'pedidos', 'lineas']
[(0, 0, 'productos', 'producto_id', 'id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'pedidos', 'pedido_id', 'id', 'NO ACTION', 'CASCADE', 'NONE')]
Rechazado: FOREIGN KEY constraint failed


## 2. Insertar con parámetros

Insertar los datos de la tienda con `executemany` y marcadores `?` y `:nombre`. Después, comparar una búsqueda que construye el SQL formateando la cadena con otra que usa parámetros, y mostrar por qué la primera es vulnerable a la inyección SQL.

In [2]:
clientes = [
    (1, "Ana Pérez", "Las Palmas", "2025-01-15"),
    (2, "Luis García", "Santa Cruz", "2025-02-03"),
    (3, "Carmen Díaz", "Las Palmas", "2025-03-21"),
    (4, "Jorge Suárez", "La Laguna", "2025-04-10"),
    (5, "Elena Ramos", "Telde", "2025-05-05"),
    (6, "Pablo Hernández", "Arucas", "2025-06-30"),
]
productos = [
    (1, "Teclado", "Informática", 35.0, 20),
    (2, "Ratón", "Informática", 15.5, 50),
    (3, "Monitor", "Informática", 189.9, 8),
    (4, "Cafetera", "Hogar", 79.0, 12),
    (5, "Tostadora", "Hogar", 29.9, 15),
    (6, "Novela", "Libros", 18.5, 40),
    (7, "Atlas", "Libros", 42.0, 5),
    (8, "Lámpara", "Hogar", 24.0, 10),
]
pedidos = [
    (1, 1, "2026-01-10"), (2, 2, "2026-01-12"), (3, 1, "2026-02-03"), (4, 3, "2026-02-14"),
    (5, 4, "2026-03-01"), (6, 2, "2026-03-08"), (7, 5, "2026-03-20"), (8, 1, "2026-04-02"),
]
lineas = [
    (1, 1, 1, 35.0), (1, 2, 2, 15.5), (2, 3, 1, 189.9), (3, 6, 3, 18.5), (3, 7, 1, 42.0),
    (4, 4, 1, 79.0), (4, 5, 1, 29.9), (5, 2, 1, 15.5), (5, 6, 2, 18.5), (6, 1, 2, 35.0),
    (6, 3, 1, 189.9), (7, 5, 2, 29.9), (8, 4, 1, 79.0), (8, 2, 1, 15.5),
]

with con:  # una sola transacción: se confirma al salir o se deshace si hay excepción
    con.executemany("INSERT INTO clientes VALUES (?, ?, ?, ?)", clientes)
    con.executemany("INSERT INTO productos VALUES (?, ?, ?, ?, ?)", productos)
    con.executemany("INSERT INTO pedidos VALUES (?, ?, ?)", pedidos)
    con.executemany(
        "INSERT INTO lineas VALUES (:pedido, :producto, :cantidad, :precio)",
        [dict(zip(["pedido", "producto", "cantidad", "precio"], l)) for l in lineas],
    )

for tabla, datos in [("clientes", clientes), ("productos", productos), ("pedidos", pedidos), ("lineas", lineas)]:
    n = con.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]  # nombre fijo, no viene del usuario
    assert n == len(datos)
    print(tabla, n)

clientes 6
productos 8
pedidos 8
lineas 14


In [3]:
def busca_inseguro(nombre):
    return con.execute(f"SELECT id, nombre FROM clientes WHERE nombre = '{nombre}'").fetchall()

def busca_seguro(nombre):
    return con.execute("SELECT id, nombre FROM clientes WHERE nombre = ?", (nombre,)).fetchall()

ataque = "x' OR '1'='1"
print("SQL generado:", f"SELECT id, nombre FROM clientes WHERE nombre = '{ataque}'")
print("Inseguro:", busca_inseguro(ataque))
print("Seguro:  ", busca_seguro(ataque))
assert len(busca_inseguro(ataque)) == len(clientes) and busca_seguro(ataque) == []

try:
    busca_inseguro("O'Brien")
except sqlite3.OperationalError as e:
    print("Un apóstrofo rompe la consulta formateada:", e)
print("Con parámetros no pasa nada:", busca_seguro("O'Brien"))

SQL generado: SELECT id, nombre FROM clientes WHERE nombre = 'x' OR '1'='1'
Inseguro: [(1, 'Ana Pérez'), (2, 'Luis García'), (3, 'Carmen Díaz'), (4, 'Jorge Suárez'), (5, 'Elena Ramos'), (6, 'Pablo Hernández')]
Seguro:   []
Un apóstrofo rompe la consulta formateada: near "Brien": syntax error
Con parámetros no pasa nada: []


## 3. Filas con nombre: `row_factory`

Usar `sqlite3.Row` para acceder a las columnas por nombre y escribir una `row_factory` propia que devuelva diccionarios.

In [4]:
cur = con.cursor()
cur.row_factory = sqlite3.Row
fila = cur.execute("SELECT * FROM productos WHERE id = ?", (3,)).fetchone()
print(fila.keys())
print(fila["nombre"], fila["precio"], fila[0], dict(fila))

def como_dict(cursor, fila):
    return {col[0]: valor for col, valor in zip(cursor.description, fila)}

cur = con.cursor()
cur.row_factory = como_dict
caros = cur.execute("SELECT nombre, precio FROM productos WHERE precio > ?", (50,)).fetchall()
print(caros)
assert caros == [{"nombre": n, "precio": p} for _, n, _, p, _ in productos if p > 50]

['id', 'nombre', 'categoria', 'precio', 'stock']
Monitor 189.9 3 {'id': 3, 'nombre': 'Monitor', 'categoria': 'Informática', 'precio': 189.9, 'stock': 8}
[{'nombre': 'Monitor', 'precio': 189.9}, {'nombre': 'Cafetera', 'precio': 79.0}]


## 4. Filtrar, ordenar y limitar

Escribir una función `tabla(sql, params)` que muestre el resultado como `DataFrame`. Obtener los productos de informática del más caro al más barato, los tres productos más caros, los que cuestan entre 20 y 50 € y la segunda página de productos (de 3 en 3) ordenados por nombre.

In [5]:
def tabla(sql, params=()):
    return pd.read_sql_query(sql, con, params=params)

display(tabla("SELECT nombre, precio FROM productos WHERE categoria = ? ORDER BY precio DESC", ("Informática",)))
display(tabla("SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 3"))
display(tabla("SELECT nombre, precio FROM productos WHERE precio BETWEEN ? AND ? ORDER BY precio", (20, 50)))
pagina = con.execute("SELECT nombre FROM productos ORDER BY nombre LIMIT 3 OFFSET 3").fetchall()
print("Página 2:", pagina)

assert pagina == [(n,) for n in sorted(p[1] for p in productos)[3:6]]
assert tabla("SELECT nombre FROM productos ORDER BY precio DESC LIMIT 3")["nombre"].tolist() == \
    [p[1] for p in sorted(productos, key=lambda p: -p[3])[:3]]

,nombre,precio
0,Monitor,189.9
1,Teclado,35.0
2,Ratón,15.5


,nombre,precio
0,Monitor,189.9
1,Cafetera,79.0
2,Atlas,42.0


,nombre,precio
0,Lámpara,24.0
1,Tostadora,29.9
2,Teclado,35.0
3,Atlas,42.0


Página 2: [('Monitor',), ('Novela',), ('Ratón',)]


## 5. `INNER JOIN`

Mostrar el detalle de todas las líneas de pedido con la fecha, el nombre del cliente, el producto, la cantidad y el importe (cantidad × precio), ordenado por pedido. Calcular después el total de cada pedido.

In [6]:
detalle = tabla('''
SELECT p.id AS pedido, p.fecha, c.nombre AS cliente, pr.nombre AS producto,
       l.cantidad, l.cantidad * l.precio AS importe
FROM pedidos p
JOIN clientes c ON c.id = p.cliente_id
JOIN lineas l ON l.pedido_id = p.id
JOIN productos pr ON pr.id = l.producto_id
ORDER BY p.id, pr.nombre
''')
display(detalle)

totales = dict(con.execute('''
SELECT pedido_id, ROUND(SUM(cantidad * precio), 2) FROM lineas GROUP BY pedido_id
''').fetchall())
print(totales)

esperado = {}
for pedido, _, cantidad, precio in lineas:
    esperado[pedido] = esperado.get(pedido, 0) + cantidad * precio
assert len(detalle) == len(lineas)
assert all(abs(totales[k] - v) < 1e-9 for k, v in esperado.items())

,pedido,fecha,cliente,producto,cantidad,importe
0,1,2026-01-10,Ana Pérez,Ratón,2,31.0
1,1,2026-01-10,Ana Pérez,Teclado,1,35.0
2,2,2026-01-12,Luis García,Monitor,1,189.9
3,3,2026-02-03,Ana Pérez,Atlas,1,42.0
4,3,2026-02-03,Ana Pérez,Novela,3,55.5
5,4,2026-02-14,Carmen Díaz,Cafetera,1,79.0
6,4,2026-02-14,Carmen Díaz,Tostadora,1,29.9
7,5,2026-03-01,Jorge Suárez,Novela,2,37.0
8,5,2026-03-01,Jorge Suárez,Ratón,1,15.5
9,6,2026-03-08,Luis García,Monitor,1,189.9


{1: 66.0, 2: 189.9, 3: 97.5, 4: 108.9, 5: 52.5, 6: 259.9, 7: 59.8, 8: 94.5}


## 6. `LEFT JOIN`

Contar los pedidos de cada cliente incluyendo a los que no tienen ninguno, y encontrar los clientes sin pedidos y los productos que nunca se han vendido. Comparar con lo que devuelve un `INNER JOIN`.

In [7]:
display(tabla('''
SELECT c.nombre, COUNT(p.id) AS pedidos
FROM clientes c LEFT JOIN pedidos p ON p.cliente_id = c.id
GROUP BY c.id ORDER BY pedidos DESC, c.nombre
'''))

interno = con.execute("SELECT COUNT(DISTINCT c.id) FROM clientes c JOIN pedidos p ON p.cliente_id = c.id").fetchone()[0]
externo = con.execute("SELECT COUNT(DISTINCT c.id) FROM clientes c LEFT JOIN pedidos p ON p.cliente_id = c.id").fetchone()[0]
print("Clientes con INNER JOIN:", interno, "con LEFT JOIN:", externo)

sin_pedidos = con.execute('''
SELECT c.nombre FROM clientes c LEFT JOIN pedidos p ON p.cliente_id = c.id WHERE p.id IS NULL
''').fetchall()
no_vendidos = con.execute('''
SELECT pr.nombre FROM productos pr LEFT JOIN lineas l ON l.producto_id = pr.id WHERE l.pedido_id IS NULL
''').fetchall()
print("Sin pedidos:", sin_pedidos, "No vendidos:", no_vendidos)

assert {n for (n,) in sin_pedidos} == {c[1] for c in clientes if c[0] not in {p[1] for p in pedidos}}
assert {n for (n,) in no_vendidos} == {p[1] for p in productos if p[0] not in {l[1] for l in lineas}}

,nombre,pedidos
0,Ana Pérez,3
1,Luis García,2
2,Carmen Díaz,1
3,Elena Ramos,1
4,Jorge Suárez,1
5,Pablo Hernández,0


Clientes con INNER JOIN: 5 con LEFT JOIN: 6
Sin pedidos: [('Pablo Hernández',)] No vendidos: [('Lámpara',)]


## 7. `GROUP BY` y `HAVING`

Calcular la facturación y el número de pedidos de cada cliente, quedarse con los que han gastado más de 200 € y obtener las unidades vendidas y la facturación de cada categoría.

In [8]:
facturacion = tabla('''
SELECT c.nombre, COUNT(DISTINCT p.id) AS pedidos, ROUND(SUM(l.cantidad * l.precio), 2) AS total
FROM clientes c
JOIN pedidos p ON p.cliente_id = c.id
JOIN lineas l ON l.pedido_id = p.id
GROUP BY c.id
HAVING total > ?
ORDER BY total DESC
''', (200,))
display(facturacion)

display(tabla('''
SELECT pr.categoria, SUM(l.cantidad) AS unidades, ROUND(SUM(l.cantidad * l.precio), 2) AS total
FROM lineas l JOIN productos pr ON pr.id = l.producto_id
GROUP BY pr.categoria ORDER BY total DESC
'''))

cliente_de = {p[0]: p[1] for p in pedidos}
nombre_de = {c[0]: c[1] for c in clientes}
gasto = {}
for pedido, _, cantidad, precio in lineas:
    nombre = nombre_de[cliente_de[pedido]]
    gasto[nombre] = gasto.get(nombre, 0) + cantidad * precio
assert set(facturacion["nombre"]) == {n for n, g in gasto.items() if g > 200}

,nombre,pedidos,total
0,Luis García,2,449.8
1,Ana Pérez,3,258.0


,categoria,unidades,total
0,Informática,9,546.8
1,Hogar,5,247.7
2,Libros,6,134.5


## 8. Subconsultas

Obtener los productos más caros que la media, los clientes que han comprado algún libro (`IN`), los productos que nadie ha pedido dos veces o más en una misma línea (`NOT EXISTS`) y el pedido con mayor importe (subconsulta en el `FROM`).

In [9]:
sobre_media = con.execute('''
SELECT nombre, precio FROM productos WHERE precio > (SELECT AVG(precio) FROM productos) ORDER BY precio
''').fetchall()
print("Más caros que la media:", sobre_media)

lectores = con.execute('''
SELECT nombre FROM clientes WHERE id IN (
    SELECT p.cliente_id FROM pedidos p JOIN lineas l ON l.pedido_id = p.id
    JOIN productos pr ON pr.id = l.producto_id WHERE pr.categoria = 'Libros'
)''').fetchall()
print("Compran libros:", lectores)

sueltos = con.execute('''
SELECT nombre FROM productos pr
WHERE NOT EXISTS (SELECT 1 FROM lineas l WHERE l.producto_id = pr.id AND l.cantidad >= 2)
ORDER BY nombre''').fetchall()
print("Nunca dos o más en una línea:", sueltos)

mayor = con.execute('''
SELECT pedido_id, total FROM (
    SELECT pedido_id, SUM(cantidad * precio) AS total FROM lineas GROUP BY pedido_id
) ORDER BY total DESC LIMIT 1''').fetchone()
print("Pedido más caro:", mayor)

media = sum(p[3] for p in productos) / len(productos)
assert sobre_media == sorted([(p[1], p[3]) for p in productos if p[3] > media], key=lambda t: t[1])
assert mayor[0] == max(esperado, key=esperado.get)

Más caros que la media: [('Cafetera', 79.0), ('Monitor', 189.9)]
Compran libros: [('Ana Pérez',), ('Jorge Suárez',)]
Nunca dos o más en una línea: [('Atlas',), ('Cafetera',), ('Lámpara',), ('Monitor',)]
Pedido más caro: (6, 259.9)


## 9. Vistas

Crear la vista `totales_pedido` (pedido, cliente, fecha y total) y usarla para calcular el ticket medio de cada cliente. Comprobar que la vista refleja al momento un pedido nuevo, sin volver a crearla.

In [10]:
con.execute('''
CREATE VIEW totales_pedido AS
SELECT p.id AS pedido, c.nombre AS cliente, p.fecha, SUM(l.cantidad * l.precio) AS total
FROM pedidos p
JOIN clientes c ON c.id = p.cliente_id
JOIN lineas l ON l.pedido_id = p.id
GROUP BY p.id
''')
display(tabla('''
SELECT cliente, COUNT(*) AS pedidos, ROUND(AVG(total), 2) AS ticket_medio
FROM totales_pedido GROUP BY cliente ORDER BY ticket_medio DESC
'''))

antes = con.execute("SELECT COUNT(*) FROM totales_pedido").fetchone()[0]
with con:
    nuevo = con.execute("INSERT INTO pedidos (cliente_id, fecha) VALUES (6, '2026-04-15')").lastrowid
    con.execute("INSERT INTO lineas VALUES (?, 8, 2, 24.0)", (nuevo,))
print(con.execute("SELECT * FROM totales_pedido WHERE pedido = ?", (nuevo,)).fetchone())
assert con.execute("SELECT COUNT(*) FROM totales_pedido").fetchone()[0] == antes + 1

,cliente,pedidos,ticket_medio
0,Luis García,2,224.9
1,Carmen Díaz,1,108.9
2,Ana Pérez,3,86.0
3,Elena Ramos,1,59.8
4,Jorge Suárez,1,52.5


(9, 'Pablo Hernández', '2026-04-15', 48.0)


## 10. Índices y `EXPLAIN QUERY PLAN`

Crear una tabla `visitas` con 200 000 visitas aleatorias a productos y medir cuánto tardan 200 consultas por producto y día antes y después de crear un índice sobre `(producto_id, fecha)`. Usar `EXPLAIN QUERY PLAN` para ver si SQLite recorre toda la tabla (`SCAN`) o usa el índice (`SEARCH`).

In [11]:
import random
import time

random.seed(0)
with con:
    con.execute("CREATE TABLE visitas (id INTEGER PRIMARY KEY, producto_id INTEGER, fecha TEXT)")
    con.executemany(
        "INSERT INTO visitas (producto_id, fecha) VALUES (?, ?)",
        ((random.randint(1, 8), f"2026-{random.randint(1, 12):02}-{random.randint(1, 28):02}") for _ in range(200_000)),
    )

sql = "SELECT COUNT(*) FROM visitas WHERE producto_id = ? AND fecha = ?"
consultas = [(random.randint(1, 8), f"2026-{random.randint(1, 12):02}-{random.randint(1, 28):02}") for _ in range(200)]

def plan(sql, params):
    return [fila[-1] for fila in con.execute("EXPLAIN QUERY PLAN " + sql, params)]

def mide():
    inicio = time.perf_counter()
    resultados = [con.execute(sql, c).fetchone()[0] for c in consultas]
    return time.perf_counter() - inicio, resultados

plan_antes = plan(sql, consultas[0])
t_antes, r_antes = mide()
con.execute("CREATE INDEX idx_visitas_producto_fecha ON visitas (producto_id, fecha)")
plan_despues = plan(sql, consultas[0])
t_despues, r_despues = mide()

print("Sin índice:", plan_antes, f"{t_antes * 1000:.1f} ms")
print("Con índice:", plan_despues, f"{t_despues * 1000:.1f} ms")
print("Plan de un JOIN:", plan("SELECT * FROM pedidos p JOIN lineas l ON l.pedido_id = p.id WHERE p.id = ?", (1,)))
assert plan_antes[0].startswith("SCAN") and "USING COVERING INDEX" in plan_despues[0]
assert r_antes == r_despues and t_despues < t_antes

Sin índice: ['SCAN visitas'] 1463.8 ms
Con índice: ['SEARCH visitas USING COVERING INDEX idx_visitas_producto_fecha (producto_id=? AND fecha=?)'] 6.5 ms
Plan de un JOIN: ['SEARCH p USING INTEGER PRIMARY KEY (rowid=?)', 'SEARCH l USING INDEX sqlite_autoindex_lineas_1 (pedido_id=?)']


## 11. Transacciones y `rollback`

Escribir `realiza_pedido(cliente_id, articulos)` que, en una sola transacción, cree el pedido, sus líneas con el precio actual y descuente el stock. Si algún producto no tiene stock suficiente, la restricción `CHECK` falla y no debe quedar nada a medias. Mostrar también un `rollback` explícito.

In [12]:
def realiza_pedido(cliente_id, articulos, fecha="2026-05-01"):
    with con:
        pedido = con.execute("INSERT INTO pedidos (cliente_id, fecha) VALUES (?, ?)", (cliente_id, fecha)).lastrowid
        for producto_id, cantidad in articulos.items():
            con.execute('''
                INSERT INTO lineas SELECT ?, id, ?, precio FROM productos WHERE id = ?
            ''', (pedido, cantidad, producto_id))
            con.execute("UPDATE productos SET stock = stock - ? WHERE id = ?", (cantidad, producto_id))
    return pedido

def estado():
    return (con.execute("SELECT COUNT(*) FROM pedidos").fetchone()[0],
            con.execute("SELECT COUNT(*) FROM lineas").fetchone()[0],
            con.execute("SELECT id, stock FROM productos ORDER BY id").fetchall())

antes = estado()
try:
    realiza_pedido(5, {2: 3, 7: 10})  # solo quedan 5 atlas
except sqlite3.IntegrityError as e:
    print("Pedido rechazado:", e)
assert estado() == antes

pedido = realiza_pedido(5, {2: 3, 7: 2})
print("Pedido", pedido, "creado. Stock:", estado()[2])
assert estado()[2][1] == (2, antes[2][1][1] - 3) and estado()[2][6] == (7, antes[2][6][1] - 2)

con.execute("UPDATE productos SET precio = precio * 2")
print("Antes del rollback:", con.execute("SELECT precio FROM productos WHERE id = 1").fetchone())
con.rollback()
print("Después del rollback:", con.execute("SELECT precio FROM productos WHERE id = 1").fetchone())
assert con.execute("SELECT precio FROM productos WHERE id = 1").fetchone() == (35.0,)

Pedido rechazado: CHECK constraint failed: stock >= 0
Pedido 10 creado. Stock: [(1, 20), (2, 47), (3, 8), (4, 12), (5, 15), (6, 40), (7, 3), (8, 10)]
Antes del rollback: (70.0,)
Después del rollback: (35.0,)


## 12. Funciones de ventana

Con funciones de ventana (`OVER`), obtener:
- el ranking de productos por unidades vendidas dentro de cada categoría (`RANK`),
- el gasto acumulado de cada cliente pedido a pedido (`SUM ... OVER`),
- los días transcurridos desde el pedido anterior del mismo cliente (`LAG`).

In [13]:
display(tabla('''
SELECT pr.categoria, pr.nombre, COALESCE(SUM(l.cantidad), 0) AS unidades,
       RANK() OVER (PARTITION BY pr.categoria ORDER BY COALESCE(SUM(l.cantidad), 0) DESC) AS puesto
FROM productos pr LEFT JOIN lineas l ON l.producto_id = pr.id
GROUP BY pr.id
ORDER BY pr.categoria, puesto
'''))

acumulado = tabla('''
SELECT cliente, fecha, ROUND(total, 2) AS total,
       ROUND(SUM(total) OVER (PARTITION BY cliente ORDER BY fecha), 2) AS acumulado,
       julianday(fecha) - julianday(LAG(fecha) OVER (PARTITION BY cliente ORDER BY fecha)) AS dias
FROM totales_pedido
ORDER BY cliente, fecha
''')
display(acumulado)

from itertools import accumulate
for cliente, grupo in acumulado.groupby("cliente"):
    assert all(abs(a - b) < 0.01 for a, b in zip(grupo["acumulado"], accumulate(grupo["total"])))
con.close()

,categoria,nombre,unidades,puesto
0,Hogar,Tostadora,3,1
1,Hogar,Cafetera,2,2
2,Hogar,Lámpara,2,2
3,Informática,Ratón,7,1
4,Informática,Teclado,3,2
5,Informática,Monitor,2,3
6,Libros,Novela,5,1
7,Libros,Atlas,3,2


,cliente,fecha,total,acumulado,dias
0,Ana Pérez,2026-01-10,66.0,66.0,NaN
1,Ana Pérez,2026-02-03,97.5,163.5,24.0
2,Ana Pérez,2026-04-02,94.5,258.0,58.0
3,Carmen Díaz,2026-02-14,108.9,108.9,NaN
4,Elena Ramos,2026-03-20,59.8,59.8,NaN
5,Elena Ramos,2026-05-01,130.5,190.3,42.0
6,Jorge Suárez,2026-03-01,52.5,52.5,NaN
7,Luis García,2026-01-12,189.9,189.9,NaN
8,Luis García,2026-03-08,259.9,449.8,55.0
9,Pablo Hernández,2026-04-15,48.0,48.0,NaN
